# Кешування в Python

Цей ноутбук показує основні ідеї кешування на коротких прикладах: мемоїзація, TTL, політики видалення, прогрів кешу та типові архітектурні патерни.

## Навіщо потрібен кеш

Кеш зберігає результат попередньої роботи, щоб не виконувати її знову. Це особливо корисно, коли обчислення, запит до бази або мережевий виклик дорогі за часом.

In [ ]:
from functools import lru_cache
import random


@lru_cache(maxsize=8)
def random_discount(user_id: int) -> int:
    print(f'Обчислюємо знижку для user_id={user_id}')
    return random.randint(5, 20)


print(random_discount(1))
print(random_discount(1))
print(random_discount(1))
print(random_discount.cache_info())

Тут функція повертає випадкове значення, але для одного й того самого `user_id` результат береться з кешу. На що звернути увагу: кешування має сенс тільки тоді, коли повторне використання старого результату справді коректне.

## Обережно з мутабельними об'єктами

Кеш зберігає не магічну копію даних, а конкретне значення, яке повернула функція. Якщо це мутабельний об'єкт, його зміни можуть неочікувано вплинути на кешований результат.

In [ ]:
from functools import lru_cache


user_profile = {'name': 'Anna', 'city': 'Kyiv'}


@lru_cache(maxsize=8)
def get_profile_snapshot(user_id: int) -> dict:
    return user_profile


first_snapshot = get_profile_snapshot(1)
print('First:', first_snapshot)

user_profile['city'] = 'Kharkiv'

second_snapshot = get_profile_snapshot(1)
print('Second:', second_snapshot)
print('Same object:', first_snapshot is second_snapshot)

Що тут відбувається: функція кешує посилання на словник. Коли словник змінюється, старий кешований об'єкт теж виглядає зміненим, бо це той самий об'єкт у пам'яті.

## Мемоїзація на прикладі Fibonacci

Мемоїзація зберігає результати викликів функції. Це добре видно на рекурсивному Fibonacci: без кешу одні й ті самі значення обчислюються багато разів.

In [ ]:
import time
from collections.abc import Callable
from memory_profiler import memory_usage


def compute_fib_plain(n: int) -> int:
    if n <= 1:
        return n
    return compute_fib_plain(n - 1) + compute_fib_plain(n - 2)


cache: dict[int, int] = {}


def compute_fib_cached(n: int) -> int:
    if n <= 1:
        return n
    if n not in cache:
        cache[n] = compute_fib_cached(n - 1) + compute_fib_cached(n - 2)
    return cache[n]


def measure(task_func: Callable, n: int):
    start = time.perf_counter()

    mem_usage = memory_usage(
        (task_func, (n,), {}),
        interval=0.01,
        max_usage=True,
        retval=False,
    )

    end = time.perf_counter()

    print(f'Час: {end - start:.4f} с')
    print(f'Памʼять: {mem_usage:.2f} MB')


measure(compute_fib_plain, 50)
measure(compute_fib_cached, 50)

У кешованій версії кожне значення `fib(n)` обчислюється один раз. Чому це важливо: інколи невеликий кеш змінює складність задачі з дуже повільної на практично миттєву.

## `lru_cache` замість ручного словника

У Python вже є готовий декоратор `lru_cache`. Він кешує результати функції та може автоматично видаляти старі значення, коли кеш заповнений.

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=128)
def compute_fib_lru(n: int) -> int:
    if n <= 1:
        return n
    return compute_fib_lru(n - 1) + compute_fib_lru(n - 2)


print(compute_fib_lru(35))
print(compute_fib_lru.cache_info())

`cache_info()` показує статистику кешу: скільки було влучань, промахів і який поточний розмір кешу. Це простий спосіб перевірити, чи кеш реально працює.

## Табуляція як альтернатива кешу

Не кожну задачу треба розв'язувати рекурсією з кешем. Іноді простіше явно побудувати таблицю значень знизу вгору.

In [ ]:
def compute_fib_tabulated(n: int) -> int:
    if n <= 1:
        return n

    table = [0] * (n + 1)
    table[1] = 1

    for i in range(2, n + 1):
        table[i] = table[i - 1] + table[i - 2]

    return table[n]


measure(compute_fib_tabulated, 50)

Табуляція не чекає повторних викликів функції, а одразу заповнює потрібні проміжні значення. Це той самий принцип уникнення повторної роботи, але в іншій формі.

## TTL: кеш із терміном життя

Іноді дані можна кешувати лише на короткий час. TTL задає, скільки секунд значення вважається свіжим.

In [ ]:
import time
from collections.abc import Callable
from functools import wraps


def ttl_cache(ttl_seconds: float) -> Callable:
    def wrap(func: Callable) -> Callable:
        cache = {}

        @wraps(func)
        def inner(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            now = time.time()

            if key in cache:
                expires_at, value = cache[key]
                if now < expires_at:
                    print('Значення взято з кешу')
                    return value
                print('Значення протермінувалося, оновлюємо')

            print('Значення не знайдено - викликаємо функцію')
            value = func(*args, **kwargs)
            cache[key] = (now + ttl_seconds, value)
            return value

        return inner

    return wrap


@ttl_cache(ttl_seconds=2.0)
def fetch_user_profile(user_id: int) -> dict:
    time.sleep(1)
    return {'user_id': user_id, 'status': 'active'}


print(fetch_user_profile(1))
print(fetch_user_profile(1))
time.sleep(2.1)
print(fetch_user_profile(1))

Перший виклик повільний, другий швидкий, а після паузи значення оновлюється. TTL корисний, коли дані змінюються, але не треба отримувати найсвіжішу версію щомілісекунди.

## Готовий TTL-кеш з `cachetools`

Для навчання корисно написати простий TTL вручну, але в робочому коді часто краще взяти готову бібліотеку. Вона вже має перевірені структури та менше шансів на дрібні помилки.

In [ ]:
import time
from cachetools import TTLCache, cached


square_cache = TTLCache(maxsize=128, ttl=2)


@cached(cache=square_cache)
def expensive_square(x: int) -> int:
    print(f'Обчислення {x} * {x}')
    time.sleep(1)
    return x * x


print('Перший виклик:', expensive_square(10))
print('Другий виклик:', expensive_square(10))
time.sleep(2.2)
print('Після завершення TTL:', expensive_square(10))

`TTLCache` поєднує обмеження за розміром і часом життя. Це типовий компроміс: ми не хочемо тримати нескінченну кількість значень і не хочемо зберігати їх надто довго.

## Застарілі дані та інвалідація

Кеш може повертати стару версію даних. Це не баг у кеші, а очікувана ціна за швидкість, якщо ми не продумали оновлення або видалення значень.

In [ ]:
import time
from cachetools import TTLCache


article_cache = TTLCache(maxsize=10, ttl=5)
database = {'article:1': {'title': 'Old title'}}


def get_article(article_id: str) -> dict:
    if article_id in article_cache:
        print('Cache hit')
        return article_cache[article_id]

    print('Cache miss -> DB read')
    value = database[article_id].copy()
    article_cache[article_id] = value
    return value


print(get_article('article:1'))
database['article:1']['title'] = 'New title'
print(get_article('article:1'))
time.sleep(5.1)
print(get_article('article:1'))

Після зміни в `database` кеш ще деякий час повертає старий заголовок. На що звернути увагу: TTL не робить дані завжди актуальними, він лише обмежує максимальний час застарівання.

## Розмір кешу та LRU

LRU означає Least Recently Used: коли кеш переповнений, видаляється елемент, до якого найдовше не зверталися. Це проста політика, яка часто добре працює для реального трафіку.

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=4)
def get_page(page_id: int) -> str:
    print(f'Loading page {page_id}')
    return f'page-{page_id}'


trace = [1, 2, 3, 4, 100, 101, 102, 103, 1, 2, 3, 4]

for page_id in trace:
    get_page(page_id)

print(get_page.cache_info())

Тут кеш занадто малий для такого патерну звернень, тому старі сторінки швидко витісняються. `hits` і `misses` допомагають побачити, чи розмір кешу відповідає навантаженню.

## Гарячі ключі та прогрів кешу

У багатьох системах невелика кількість ключів отримує більшість запитів. Такі ключі можна знайти в логах і завантажити в кеш заздалегідь.

In [ ]:
from collections import Counter


requests = [
    '/home',
    '/home',
    '/home',
    '/search',
    '/product/1',
    '/home',
    '/search',
    '/product/2',
    '/home',
    '/product/1',
    '/checkout',
    '/home',
    '/search',
    '/home',
    '/product/3',
]

counter = Counter(requests)
print(counter)

hot_keys = [key for key, _ in counter.most_common(2)]
print('Warm-up candidates:', hot_keys)

Цей приклад показує просту ідею cache warm-up. Якщо ми знаємо популярні ключі, можна підготувати кеш до пікового навантаження ще до перших запитів користувачів.

## Hit rate на Zipf-навантаженні

Запити в реальних системах часто нерівномірні: популярні елементи запитують набагато частіше за інші. Розподіл Zipf допомагає змоделювати таку ситуацію.

In [ ]:
from collections import Counter, OrderedDict

import matplotlib.pyplot as plt
import numpy as np


MPLCONFIGDIR = '/tmp/matplotlib'
XDG_CACHE_HOME = '/tmp'


def generate_zipf_trace(num_items: int, num_requests: int, skew: float, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    ranks = rng.zipf(skew, size=num_requests)
    ranks = np.clip(ranks, 1, num_items)
    return ranks - 1


def compute_hotset_n90(requests: np.ndarray) -> int:
    frequency = Counter(requests)
    counts = np.array([count for _, count in frequency.most_common()])
    cumulative = np.cumsum(counts)
    threshold = 0.9 * len(requests)
    return int(np.searchsorted(cumulative, threshold) + 1)


def simulate_lru_hit_rate(trace: np.ndarray, cache_capacity: int) -> float:
    if cache_capacity <= 0:
        return 0.0

    cache = OrderedDict()
    hits = 0

    for item in trace:
        if item in cache:
            hits += 1
            cache.move_to_end(item)
        else:
            cache[item] = True
            if len(cache) > cache_capacity:
                cache.popitem(last=False)

    return hits / len(trace)


requests_trace = generate_zipf_trace(num_items=512, num_requests=20_000, skew=1.2)
n90 = compute_hotset_n90(requests_trace)

cache_sizes = np.unique(np.concatenate([np.arange(0, 65), np.linspace(64, 512, 32, dtype=int)]))
hit_rates = np.array([simulate_lru_hit_rate(requests_trace, size) for size in cache_sizes])

plt.figure(figsize=(8, 4))
plt.plot(cache_sizes, hit_rates, marker='o', label='Hit rate')
plt.axvline(n90, linestyle='--', color='orange', label=f'N90 = {n90}')
plt.xlabel('Розмір кешу, елементів')
plt.ylabel('Hit rate')
plt.title('Ефективність LRU кешу для Zipf-навантаження')
plt.grid(True)
plt.legend()
plt.show()

Графік показує, як збільшення кешу впливає на частку влучань. Після певного розміру користь від додаткової пам'яті зростає повільніше, і це важливо для вибору практичного розміру кешу.

## Cache-through: читання і запис через кеш

У cache-through додаток працює через шар кешу. При читанні кеш сам завантажує дані з бази, а при записі оновлює і кеш, і базу.

In [ ]:
cache = {}
database = {
    'user:1': {'name': 'Anna', 'status': 'active'},
}


def cache_read(key: str) -> dict | None:
    if key in cache:
        print('Cache: hit')
        return cache[key]

    print('Cache: miss -> load from database')
    value = database.get(key)
    if value is not None:
        cache[key] = value.copy()
    return cache.get(key)


def cache_write(key: str, value: dict) -> None:
    print('Cache: write')
    cache[key] = value.copy()

    print('Database: write')
    database[key] = value.copy()


def get_user(user_id: int) -> dict | None:
    return cache_read(f'user:{user_id}')


def update_user_status(user_id: int, new_status: str) -> None:
    key = f'user:{user_id}'
    current = cache_read(key)
    if current is None:
        return

    updated = current.copy()
    updated['status'] = new_status
    cache_write(key, updated)


print('=== First read ===')
print(get_user(1))

print('\n=== Second read ===')
print(get_user(1))

print('\n=== Write ===')
update_user_status(1, 'blocked')

print('\n=== Read after write ===')
print(get_user(1))

print('\nCache state:', cache)
print('Database state:', database)

Цей підхід зменшує ризик розсинхронізації, бо запис проходить через один контрольований шлях. Але в реальних системах все одно треба думати про помилки, транзакції та порядок оновлень.

## Cache Aside та проблема Cache stampede

Якщо багато потоків одночасно бачать cache miss, вони можуть всі разом піти в базу. Це називають cache stampede або thundering herd.

In [ ]:
import threading
import time


database = {
    'product:1': {'name': 'Laptop', 'price': 50_000},
}

cache_without_lock = {}
cache_with_lock = {}
cache_lock = threading.Lock()


def get_product_without_lock(product_id: int) -> dict | None:
    key = f'product:{product_id}'

    if key in cache_without_lock:
        print(f'{threading.current_thread().name}: cache hit')
        return cache_without_lock[key]

    print(f'{threading.current_thread().name}: cache miss -> DB read')
    time.sleep(1)

    value = database.get(key)
    if value is not None:
        cache_without_lock[key] = value.copy()
    return value


def get_product_with_lock(product_id: int) -> dict | None:
    key = f'product:{product_id}'

    if key in cache_with_lock:
        print(f'{threading.current_thread().name}: cache hit')
        return cache_with_lock[key]

    with cache_lock:
        if key in cache_with_lock:
            print(f'{threading.current_thread().name}: cache hit after lock')
            return cache_with_lock[key]

        print(f'{threading.current_thread().name}: cache miss -> DB read')
        time.sleep(1)

        value = database.get(key)
        if value is not None:
            cache_with_lock[key] = value.copy()
        return value


def run_workers(label: str, get_product_func) -> None:
    def worker() -> None:
        product = get_product_func(1)
        print(f'{threading.current_thread().name}: got {product}')

    threads = [threading.Thread(target=worker, name=f'T{i}') for i in range(5)]

    print(label)
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()


run_workers('=== Without lock ===', get_product_without_lock)
run_workers('\n=== With lock ===', get_product_with_lock)

Без lock кілька потоків одночасно виконують дорогий запит. З lock тільки один потік завантажує значення, а інші після цього читають уже готовий кеш.

## Cache-ahead і write-behind

Cache-ahead завантажує важливі дані до того, як їх попросять користувачі. Write-behind спочатку пише в кеш, а базу оновлює пізніше, тому він швидкий, але ризикований.

In [ ]:
database = {
    'top:articles': ['Python', 'Caching', 'NumPy'],
}
cache = {}


def cache_ahead_load() -> None:
    print('Preloading hot data into cache...')
    cache['top:articles'] = database['top:articles'].copy()


def read_top_articles() -> list[str] | None:
    print('App reads from cache')
    return cache.get('top:articles')


def write_behind_with_failure(key: str, value: list[str]) -> None:
    print('Write to cache first')
    cache[key] = value.copy()

    print('Response returned before DB sync')
    print('Crash before database write')
    raise RuntimeError('DB sync did not happen')  # можна протестувати без цього


print('=== Cache Ahead ===')
cache_ahead_load()
print('Read result:', read_top_articles())

print('\n=== Write-behind ===')
try:
    write_behind_with_failure(
        'top:articles',
        ['Python', 'Caching', 'NumPy', 'JAX'],
    )
except RuntimeError as error:
    print('Error:', error)

print('\nFinal state:')
print('Cache   :', cache)
print('Database:', database)

Після помилки кеш і база мають різний стан. Це головна ідея прикладу: write-behind може зменшити latency, але потребує черг, повторних спроб і чіткої стратегії відновлення.

## Підсумок

Кешування - це спосіб не виконувати одну й ту саму дорогу роботу повторно, а зберігати вже обчислений результат і швидко повертати його вдруге. Воно може суттєво прискорити програму, зменшити кількість запитів до бази, API чи диска, але тільки тоді, коли повторне використання старого результату справді коректне. Під час роботи з кешем важливо пам’ятати про його розмір, час життя даних, політику витіснення та ризик застарілих значень. Тому кеш - це не просто оптимізація, а інструмент, який треба застосовувати обережно і лише там, де він дійсно виправданий.